# DistilBERT Fine-Tuning for Malicious IP Detection
**Updated:** Progress bars disabled to avoid widget errors.

This notebook:
1. Loads your `fine_tuning_dataset_2.jsonl` (with `input`/`output`).
2. Maps "malicious"/"benign" labels to `1`/`0`.
3. Tokenizes with **DistilBERT** tokenizer.
4. Fine-tunes on Apple MPS (if available).
5. Evaluates on a held-out set and saves the best model.


In [2]:
# Install dependencies (run once)
!pip install --upgrade pip
!pip install transformers datasets scikit-learn torch torchvision accelerate

In [4]:
# 1️⃣ Imports & Config
import os
import numpy as np
import torch
from datasets import load_dataset, disable_progress_bar
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Disable interactive progress bars
disable_progress_bar()

# Configuration
DATASET_JSONL = "fine_tuning_dataset_2.jsonl"  # your JSONL file
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "./distil_model_ft"
NUM_LABELS = 2  # binary

# Device setup (MPS on Apple Silicon)
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [8]:
# 2️⃣ Load & Preprocess the Dataset
raw = load_dataset(
    "json",
    data_files={"train": DATASET_JSONL},
    split="train"
)
dataset = raw.train_test_split(test_size=0.1, seed=42)
def map_labels(example):
    lab = example["output"].strip().lower()
    return {"label": 1 if lab == "malicious" else 0}
dataset = dataset.map(map_labels)
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
def tokenize_fn(ex): return tokenizer(ex["input"], truncation=True)
dataset = dataset.map(tokenize_fn, batched=True)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Print summary instead of relying on widget
print(dataset)
print("\nExample training entry:")
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'label', 'input_ids', 'attention_mask'],
        num_rows: 8725
    })
    test: Dataset({
        features: ['input', 'output', 'label', 'input_ids', 'attention_mask'],
        num_rows: 970
    })
})

Example training entry:
{'label': tensor(0), 'input_ids': tensor([  101,  2335, 15464,  2361,  1024,  1064,  5034,  6895,  2361,  1024,
         4724, 19317,  2620,  1012,  1014,  1064,  5034, 21906, 11589,  1024,
         9388,  2410,  2539,  1024,  2484,  1024,  4583, 19723, 17084,  1011,
         2773, 20110,  7020, 14945,  1031, 26267,  2581,  2549,  1033,  1024,
        19528,  5310,  1057,  8569,  3372,  2226,  2013, 16087,  1012,  6109,
         1012, 17403,  1012, 11816,  3417,  4724, 19317,  2620,  1064,  2406,
         1024,  4361,  1064,  2103,  1024,  1063,  1000,  8840,  2078,  1000,
         1024,  1011,  6535,  1012,  4029, 21926,  1064,  3295,  1024,  2474,
         2102,  1024,  4724,  1012,  3515,

In [10]:
# 3️⃣ Define Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1}

In [16]:
# 4️⃣ Load Model & Setup Trainer
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model.to(device)

data_collator = DataCollatorWithPadding(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="steps",     # run evaluation every N steps
    save_strategy="steps",     # save checkpoints every N steps
    eval_steps=200,            # must divide save_steps
    save_steps=200,            # must match eval_steps
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/nx/ncjqnbc91tq32jr3cgr_7m8c0000gp/T/ipykernel_4228/1901335102.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [18]:
# 5️⃣ Train!
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model & tokenizer saved to {OUTPUT_DIR}")

/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
200,0.000100,0.000043,1.000000,0.000000,0.000000,0.000000
400,0.000000,0.000005,1.000000,0.000000,0.000000,0.000000
600,0.000000,0.000002,1.000000,0.000000,0.000000,0.000000
800,0.000000,0.000001,1.000000,0.000000,0.000000,0.000000
1000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
1200,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
1400,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
1600,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
1800,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000


/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/d

✅ Model & tokenizer saved to ./distil_model_ft
